# Critical Path Method (CPM)

Quaderno del corso **Ottimizzazione su reti** — Giampaolo Liuzzi, Sapienza Università di Roma, DIAG.

Le celle possono essere eseguite in ordine. Gli esempi usano NetworkX e Matplotlib.

## Modello activity-on-node

Ogni nodo rappresenta un'attività e ogni arco un vincolo di precedenza. Calcoliamo tempi al più presto, al più tardi e margini totali.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt


In [ ]:
D=nx.DiGraph()
durate={'A':3,'B':2,'C':4,'D':3,'E':2,'F':1}
D.add_nodes_from((a,{'durata':d}) for a,d in durate.items())
D.add_edges_from([('A','C'),('B','C'),('B','D'),('C','E'),('D','E'),('E','F')])
assert nx.is_directed_acyclic_graph(D)


In [ ]:
ES={}; EF={}
for a in nx.topological_sort(D):
    ES[a]=max((EF[p] for p in D.predecessors(a)),default=0)
    EF[a]=ES[a]+durate[a]
T=max(EF.values())
LF={}; LS={}
for a in reversed(list(nx.topological_sort(D))):
    LF[a]=min((LS[s] for s in D.successors(a)),default=T)
    LS[a]=LF[a]-durate[a]
slack={a:LS[a]-ES[a] for a in D}
print('Durata progetto:',T)
for a in D: print(a, 'ES',ES[a],'EF',EF[a],'LS',LS[a],'LF',LF[a],'margine',slack[a])


In [ ]:
critiche=[a for a in D if slack[a]==0]
print('Attività critiche:',critiche)
nx.set_node_attributes(D, ES, 'livello')
pos=nx.multipartite_layout(D,subset_key='livello',align='horizontal')
colors=['#ffcccc' if a in critiche else '#dbeeff' for a in D]
labels={a:f'{a}\n{durate[a]}' for a in D}
nx.draw_networkx(D,pos,labels=labels,node_color=colors,node_size=1200,arrows=True)
plt.title('Rete CPM; attività critiche in rosso'); plt.axis('off')


## Esercizi

1. Identificare il cammino critico e verificarne la durata.
2. Aumentare di due unità la durata di `D` e ripetere i calcoli.
3. Stabilire di quanto può ritardare `B` senza ritardare il progetto.
4. Aggiungere una nuova attività `G` successiva a `C` e precedente a `F`.